# 2 · Feature engineering (computed per timeframe, no cross-TF leakage)

| TF | Features |
|----|----------|
| D1 | HH/HL/LH/LL swing structure, weekly pivot S/R |
| H4 | Swing H/L, BOS, ATR, BB width |
| H1 | Session flags, EMA cross, MACD |
| M15 | ATR, engulfing / pin bar, candle body ratio |

All rolling/EWM ops are causal (no `shift(-1)`, no `center=True`) — each row only uses that bar and earlier ones.

**Exception:** `swing_points()` is a *bilateral* fractal — it looks `lookback` bars into the future to confirm a pivot, and `filter_alternating()` post-processes across the whole series. `swing_high`/`swing_low`/`is_hh`/`is_lh`/`is_hl`/`is_ll` (D1) and `swing_high`/`swing_low`/`bos_up`/`bos_down` (H4) therefore carry lookahead and are for **chart visualization only** — do not feed them into training as-is. If needed as a model feature, only use a swing flagged at bar `i` once `i <= t - lookback` (i.e. shift by `lookback` bars before use).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config.settings import DATA_PROCESSED, DATA_FEATURES

SYMBOL = "GBPUSD"
TIMEFRAMES = ["D1", "H4", "H1", "M15"]

clean = {tf: pd.read_parquet(DATA_PROCESSED / f"{SYMBOL}_{tf}.parquet") for tf in TIMEFRAMES}
for tf, df in clean.items():
    print(f"{tf}: {len(df)} rows  ({df['datetime'].iloc[0]} .. {df['datetime'].iloc[-1]})")

D1: 3783 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H4: 22680 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H1: 90426 rows  (2012-01-11 01:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
M15: 356499 rows  (2012-01-11 01:30:00+00:00 .. 2026-07-10 00:00:00+00:00)


## Shared helpers

In [2]:
def atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Average true range (Wilder), causal."""
    prev_close = df["close"].shift(1)
    tr = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - prev_close).abs(),
            (df["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    return tr.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()


def swing_points(df: pd.DataFrame, lookback: int = 5):
    """
    True bilateral fractal swing detection.
    
    Bar i is confirmed as a swing HIGH at bar i+lookback.
    In your feature dataframe, the flag sits at bar i (the pivot),
    but when building observations at bar t, you must only use
    swing points where i <= t - lookback (confirmed swings only).
    
    Last `lookback` bars will always be 0 — unconfirmable.
    """
    highs = df["high"].values
    lows  = df["low"].values
    n = len(df)
    
    sh = np.zeros(n, dtype=bool)
    sl = np.zeros(n, dtype=bool)
    
    for i in range(lookback, n - lookback):
        # Left window: bars i-lookback to i-1
        # Right window: bars i+1 to i+lookback
        if highs[i] > highs[i-lookback:i].max() and \
           highs[i] > highs[i+1:i+lookback+1].max():
            sh[i] = True
        
        if lows[i] < lows[i-lookback:i].min() and \
           lows[i] < lows[i+1:i+lookback+1].min():
            sl[i] = True
    
    return pd.Series(sh, index=df.index), pd.Series(sl, index=df.index)


def filter_alternating(
    df: pd.DataFrame, swing_high: pd.Series, swing_low: pd.Series
) -> tuple[pd.Series, pd.Series]:
    """
    Enforce strict H/L alternation. When two swing highs occur with no swing
    low between them, keep only the higher one (and symmetrically for lows) —
    drops the noisy intermediate pivots that a pure fractal check lets through
    during choppy stretches.
    """
    points = sorted(
        [(i, "H", df["high"].iat[i]) for i in np.flatnonzero(swing_high.values)]
        + [(i, "L", df["low"].iat[i]) for i in np.flatnonzero(swing_low.values)]
    )

    kept = []
    for i, kind, price in points:
        if kept and kept[-1][1] == kind:
            # same side as previous kept point — keep only the more extreme one
            if (kind == "H" and price >= kept[-1][2]) or (kind == "L" and price <= kept[-1][2]):
                kept[-1] = (i, kind, price)
            continue
        kept.append((i, kind, price))

    sh_idx = [i for i, kind, _ in kept if kind == "H"]
    sl_idx = [i for i, kind, _ in kept if kind == "L"]

    out_high = pd.Series(False, index=df.index)
    out_low = pd.Series(False, index=df.index)
    out_high.iloc[sh_idx] = True
    out_low.iloc[sl_idx] = True
    return out_high, out_low

## D1 · HH/HL/LH/LL swing structure, weekly pivot S/R

In [3]:
def features_d1(df: pd.DataFrame, lookback: int = 5) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    swing_high, swing_low = swing_points(df, lookback)
    swing_high, swing_low = filter_alternating(df, swing_high, swing_low)
    last_swing_high = df["high"].where(swing_high).ffill()
    last_swing_low = df["low"].where(swing_low).ffill()
    prev_swing_high = last_swing_high.shift(1)
    prev_swing_low = last_swing_low.shift(1)

    out["swing_high"] = swing_high.astype(int)
    out["swing_low"] = swing_low.astype(int)
    out["is_hh"] = (swing_high & (last_swing_high > prev_swing_high)).astype(int)
    out["is_lh"] = (swing_high & (last_swing_high < prev_swing_high)).astype(int)
    out["is_hl"] = (swing_low & (last_swing_low > prev_swing_low)).astype(int)
    out["is_ll"] = (swing_low & (last_swing_low < prev_swing_low)).astype(int)

    # weekly pivot S/R computed from the *prior completed* week only (no lookahead)
    week = df["datetime"].dt.tz_convert(None).dt.to_period("W-SUN")
    weekly = df.groupby(week).agg(
        w_high=("high", "max"), w_low=("low", "min"), w_close=("close", "last")
    )
    weekly_prev = weekly.shift(1)
    pivot = (weekly_prev["w_high"] + weekly_prev["w_low"] + weekly_prev["w_close"]) / 3
    r1 = 2 * pivot - weekly_prev["w_low"]
    s1 = 2 * pivot - weekly_prev["w_high"]
    r2 = pivot + (weekly_prev["w_high"] - weekly_prev["w_low"])
    s2 = pivot - (weekly_prev["w_high"] - weekly_prev["w_low"])

    pivot_df = pd.DataFrame({"pivot": pivot, "r1": r1, "s1": s1, "r2": r2, "s2": s2})
    mapped = week.map(pivot_df.to_dict("index")).apply(pd.Series)
    out[["pivot", "r1", "s1", "r2", "s2"]] = mapped.values
    out["dist_to_pivot"] = (df["close"] - out["pivot"]) / df["close"]

    return out


feat_d1 = features_d1(clean["D1"])
feat_d1.tail()

,datetime,swing_high,swing_low,is_hh,is_lh,is_hl,is_ll,pivot,r1,s1,r2,s2,dist_to_pivot
3778,2026-07-06 00:00:00+00:00,0,0,0,0,0,0,1.330863,1.342377,1.323287,1.349953,1.311773,0.006351
3779,2026-07-07 00:00:00+00:00,0,0,0,0,0,0,1.330863,1.342377,1.323287,1.349953,1.311773,0.002807
3780,2026-07-08 00:00:00+00:00,0,0,0,0,0,0,1.330863,1.342377,1.323287,1.349953,1.311773,0.006329
3781,2026-07-09 00:00:00+00:00,0,0,0,0,0,0,1.330863,1.342377,1.323287,1.349953,1.311773,0.007796
3782,2026-07-10 00:00:00+00:00,0,0,0,0,0,0,1.330863,1.342377,1.323287,1.349953,1.311773,0.007478


## H4 · Swing H/L, BOS, ATR, BB width

In [4]:
def features_h4(
    df: pd.DataFrame, lookback: int = 5, atr_period: int = 14, bb_period: int = 20
) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    swing_high, swing_low = swing_points(df, lookback)
    swing_high, swing_low = filter_alternating(df, swing_high, swing_low)
    last_swing_high = df["high"].where(swing_high).ffill()
    last_swing_low = df["low"].where(swing_low).ffill()

    out["swing_high"] = swing_high.astype(int)
    out["swing_low"] = swing_low.astype(int)

    # BOS: close breaks the most recent *confirmed prior* swing level
    prior_swing_high = last_swing_high.shift(1)
    prior_swing_low = last_swing_low.shift(1)
    out["bos_up"] = (df["close"] > prior_swing_high).astype(int)
    out["bos_down"] = (df["close"] < prior_swing_low).astype(int)

    out["atr"] = atr(df, atr_period)
    out["atr_pct"] = out["atr"] / df["close"]

    bb_mid = df["close"].rolling(bb_period, min_periods=bb_period).mean()
    bb_std = df["close"].rolling(bb_period, min_periods=bb_period).std()
    out["bb_width"] = (4 * bb_std) / bb_mid

    return out


feat_h4 = features_h4(clean["H4"])
feat_h4.tail()

,datetime,swing_high,swing_low,bos_up,bos_down,atr,atr_pct,bb_width
22675,2026-07-09 08:00:00+00:00,0,0,0,0,0.002856,0.002134,0.006615
22676,2026-07-09 12:00:00+00:00,0,0,0,0,0.002868,0.002139,0.006691
22677,2026-07-09 16:00:00+00:00,0,0,0,0,0.002785,0.002078,0.006547
22678,2026-07-09 20:00:00+00:00,0,0,0,0,0.002735,0.002039,0.006829
22679,2026-07-10 00:00:00+00:00,0,0,0,0,0.002572,0.001918,0.007029


## H1 · Session flags, EMA cross, MACD

In [5]:
# UTC session hours (approximate, DST-naive — good enough for session flags)
SESSIONS = {
    "asian": (0, 8),
    "london": (7, 16),
    "ny": (12, 21),
}


def features_h1(
    df: pd.DataFrame, ema_fast: int = 12, ema_slow: int = 26, macd_signal: int = 9
) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    hour = df["datetime"].dt.hour
    for name, (start, end) in SESSIONS.items():
        out[f"session_{name}"] = ((hour >= start) & (hour < end)).astype(int)
    out["session_overlap_ldn_ny"] = (out["session_london"] & out["session_ny"]).astype(int)

    ema_f = df["close"].ewm(span=ema_fast, adjust=False, min_periods=ema_fast).mean()
    ema_s = df["close"].ewm(span=ema_slow, adjust=False, min_periods=ema_slow).mean()
    out["ema_fast"] = ema_f
    out["ema_slow"] = ema_s
    ema_diff_sign = np.sign(ema_f - ema_s)
    out["ema_cross_up"] = ((ema_diff_sign == 1) & (ema_diff_sign.shift(1) <= 0)).astype(int)
    out["ema_cross_down"] = ((ema_diff_sign == -1) & (ema_diff_sign.shift(1) >= 0)).astype(int)

    macd_line = ema_f - ema_s
    macd_sig = macd_line.ewm(span=macd_signal, adjust=False, min_periods=macd_signal).mean()
    out["macd"] = macd_line
    out["macd_signal"] = macd_sig
    out["macd_hist"] = macd_line - macd_sig

    return out


feat_h1 = features_h1(clean["H1"])
feat_h1.tail()

,datetime,session_asian,session_london,session_ny,session_overlap_ldn_ny,ema_fast,ema_slow,ema_cross_up,ema_cross_down,macd,macd_signal,macd_hist
90421,2026-07-09 20:00:00+00:00,0,0,1,0,1.340542,1.339860,0,0,0.000682,0.000720,-0.000038
90422,2026-07-09 21:00:00+00:00,0,0,0,0,1.340478,1.339880,0,0,0.000599,0.000696,-0.000097
90423,2026-07-09 22:00:00+00:00,0,0,0,0,1.340585,1.339975,0,0,0.000609,0.000679,-0.000069
90424,2026-07-09 23:00:00+00:00,0,0,0,0,1.340698,1.340075,0,0,0.000623,0.000668,-0.000045
90425,2026-07-10 00:00:00+00:00,1,0,0,0,1.340727,1.340135,0,0,0.000592,0.000652,-0.000060


## M15 · Engulfing / pin bar, candle body ratio

In [6]:
def features_m15(df: pd.DataFrame, pin_wick_ratio: float = 2.0, atr_period: int = 14) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    out["atr"] = atr(df, atr_period)
    out["atr_pct"] = out["atr"] / df["close"]

    body = (df["close"] - df["open"]).abs()
    rng = (df["high"] - df["low"]).replace(0, np.nan)
    upper_wick = df["high"] - df[["open", "close"]].max(axis=1)
    lower_wick = df[["open", "close"]].min(axis=1) - df["low"]

    out["body_ratio"] = (body / rng).fillna(0.0)
    out["upper_wick_ratio"] = (upper_wick / rng).fillna(0.0)
    out["lower_wick_ratio"] = (lower_wick / rng).fillna(0.0)

    prev_open = df["open"].shift(1)
    prev_close = df["close"].shift(1)
    bullish = df["close"] > df["open"]
    prev_bearish = prev_close < prev_open

    out["bullish_engulf"] = (
        bullish & prev_bearish & (df["close"] >= prev_open) & (df["open"] <= prev_close)
    ).astype(int)
    out["bearish_engulf"] = (
        (~bullish) & (~prev_bearish) & (df["open"] >= prev_close) & (df["close"] <= prev_open)
    ).astype(int)

    out["bull_pin_bar"] = ((lower_wick >= pin_wick_ratio * body) & (lower_wick > upper_wick)).astype(int)
    out["bear_pin_bar"] = ((upper_wick >= pin_wick_ratio * body) & (upper_wick > lower_wick)).astype(int)

    return out


feat_m15 = features_m15(clean["M15"])
feat_m15.tail()

,datetime,atr,atr_pct,body_ratio,upper_wick_ratio,lower_wick_ratio,bullish_engulf,bearish_engulf,bull_pin_bar,bear_pin_bar
356494,2026-07-09 23:00:00+00:00,0.000570,0.000425,0.555556,0.222222,0.222222,0,0,0,0
356495,2026-07-09 23:15:00+00:00,0.000542,0.000404,0.866667,0.000000,0.133333,0,0,0,0
356496,2026-07-09 23:30:00+00:00,0.000518,0.000386,0.857143,0.047619,0.095238,1,0,0,0
356497,2026-07-09 23:45:00+00:00,0.000495,0.000369,0.300000,0.000000,0.700000,0,0,1,0
356498,2026-07-10 00:00:00+00:00,0.000493,0.000367,0.913043,0.043478,0.043478,0,0,0,0


## Save (each TF's features saved standalone, keyed by its own datetime — no merge)

In [7]:
feat_by_tf = {"D1": feat_d1, "H4": feat_h4, "H1": feat_h1, "M15": feat_m15}

for tf, df in feat_by_tf.items():
    path = DATA_FEATURES / f"{SYMBOL}_{tf}_features.parquet"
    df.to_parquet(path, index=False)
    print(f"[SAVE] {path.relative_to(DATA_FEATURES.parent.parent)}: {len(df)} rows, {df.shape[1]} cols")

[SAVE] data/features/GBPUSD_D1_features.parquet: 3783 rows, 13 cols
[SAVE] data/features/GBPUSD_H4_features.parquet: 22680 rows, 8 cols
[SAVE] data/features/GBPUSD_H1_features.parquet: 90426 rows, 12 cols
[SAVE] data/features/GBPUSD_M15_features.parquet: 356499 rows, 10 cols
